# LeetCode 2917 — K-or of an Array bằng Python

Link đề: https://leetcode.com/problems/k-or-of-an-array/ · Độ khó: **Easy** · Chủ đề: Array, Bit Manipulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def findKOr(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **K-or** là số mà bit thứ `i` của nó bằng `1` **khi và chỉ khi có ít nhất `k` phần tử** của `nums` có bit thứ `i` bằng `1`.
2. Hiểu cách **"nhìn từng bit"**: biểu diễn nhị phân, **bit thứ `i`**, và các thao tác `(x >> i) & 1` (đọc bit), `1 << i` (tạo mặt nạ bit), `|` (bật bit).
3. Đi từ cách **đếm từng bit qua mọi phần tử** → **chỉ duyệt các bit đang bật của mỗi số** → **xem "cột bit" bằng chuỗi nhị phân và `zip`** → **các trường hợp đặc biệt `k = 1` (OR) và `k = n` (AND)**.
4. Hiểu K-or là **tổng quát hóa** của OR (`k = 1`) và AND (`k = n`).
5. Tránh các lỗi kinh điển: chỉ duyệt 30 bit thay vì 31, dùng `> k` thay vì `>= k`, cộng `x & (1 << b)` (là `2^b` chứ không phải `1`) vào bộ đếm, cộng `b` thay vì `1 << b` vào kết quả, và chuỗi nhị phân không đệm cùng độ dài khi dùng `zip`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. **K-or** của `nums` là một số nguyên thỏa: bit thứ `i` (đếm từ `0`, bit thấp nhất) của K-or bằng `1` **khi và chỉ khi** có **ít nhất `k`** phần tử trong `nums` mà bit thứ `i` của chúng bằng `1`.

Hãy trả về K-or của `nums`.

**Ví dụ 1**

- Input: `nums = [7, 12, 9, 8, 9, 15]`, `k = 4`
- Output: `9`
- Giải thích: số phần tử có bit `0` bằng `1` là 4 (`7, 9, 9, 15`); bit `1`: 2; bit `2`: 3; bit `3`: 5 (`12, 9, 8, 9, 15`). Chỉ bit `0` và bit `3` có ít nhất `k = 4` phần tử, nên K-or `= 2⁰ + 2³ = 1 + 8 = 9`.

**Ví dụ 2**

- Input: `nums = [2, 12, 1, 11, 4, 5]`, `k = 6`
- Output: `0`
- Giải thích: `k = 6` bằng số phần tử, nên K-or chính là **AND** của mọi phần tử, bằng `0`.

**Ví dụ 3**

- Input: `nums = [10, 8, 5, 9, 11, 6, 8]`, `k = 1`
- Output: `15`
- Giải thích: `k = 1` nên K-or chính là **OR** của mọi phần tử: `10 | 8 | 5 | 9 | 11 | 6 | 8 = 15`.

**Ràng buộc:** `1 <= nums.length <= 50`; `0 <= nums[i] < 2^31`; `1 <= k <= nums.length`.

In [1]:
from typing import List
from functools import reduce
from operator import and_, or_
import random

examples = [
    ([7, 12, 9, 8, 9, 15], 4, 9),
    ([2, 12, 1, 11, 4, 5], 6, 0),
    ([10, 8, 5, 9, 11, 6, 8], 1, 15),
]
for nums, k, expected in examples:
    print("nums =", nums, "k =", k, "-> kỳ vọng", expected)

nums = [7, 12, 9, 8, 9, 15] k = 4 -> kỳ vọng 9
nums = [2, 12, 1, 11, 4, 5] k = 6 -> kỳ vọng 0
nums = [10, 8, 5, 9, 11, 6, 8] k = 1 -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1. Biểu diễn nhị phân và "bit thứ `i`"

Một số nguyên được viết bằng các **bit** `0`/`1`. **Bit thứ `i`** (đếm từ bên **phải**, bắt đầu từ `0`) có **trọng số `2ⁱ`**. Ví dụ `12 = 1100₂`: bit 0 = 0, bit 1 = 0, bit 2 = 1, bit 3 = 1, nên `12 = 1·4 + 1·8`.

In [2]:
x = 12
print(bin(x))                                   # 0b1100
for i in range(5):
    print(f"bit {i} của {x} =", (x >> i) & 1)    # đọc từng bit

0b1100
bit 0 của 12 = 0
bit 1 của 12 = 0
bit 2 của 12 = 1
bit 3 của 12 = 1
bit 4 của 12 = 0


### 3.2. Đọc bit: `(x >> i) & 1`

- `x >> i`: **dịch phải `i` bit**, đưa bit thứ `i` xuống vị trí thấp nhất;
- `& 1`: giữ lại **chỉ bit thấp nhất** (kết quả `0` hoặc `1`).

Ghép lại, `(x >> i) & 1` là **giá trị của bit thứ `i`** của `x`. Nên viết **đủ ngoặc** cho rõ.

In [3]:
print((9 >> 0) & 1, (9 >> 1) & 1, (9 >> 2) & 1, (9 >> 3) & 1)    # 9 = 1001 -> 1 0 0 1

1 0 0 1


### 3.3. Mặt nạ bit `1 << i` và phép `&`

`1 << i` là số **chỉ có bit `i` bằng `1`** (giá trị `2ⁱ`). `x & (1 << i)` cho **`2ⁱ` nếu bit `i` của `x` bật, ngược lại `0`**: kết quả **không phải `0/1`** mà là `0` hoặc `2ⁱ`. Dùng trong điều kiện `if` thì `2ⁱ` vẫn "đúng", nhưng **cộng vào bộ đếm sẽ sai**.

In [4]:
print(1 << 3)                       # 8
print(12 & (1 << 3))                # 8: bit 3 của 12 bật -> 8, không phải 1
print(12 & (1 << 1))                # 0: bit 1 của 12 tắt
print(bool(12 & (1 << 3)), bool(12 & (1 << 1)))     # True False

8
8
0
True False


### 3.4. Phép OR `|` để "bật" bit trong kết quả

`a | b` có bit `i` bằng `1` nếu **bit `i` của `a` hoặc của `b`** bằng `1`. Để **bật bit `b`** trong kết quả: `ans |= 1 << b`. Ta dùng để dựng K-or: với mỗi bit đủ điều kiện, bật nó trong `ans`.

In [5]:
ans = 0
for b in [0, 3]:
    ans |= 1 << b
    print(f"bật bit {b}: ans = {ans} = {bin(ans)}")
print(5 | 2, bin(5), bin(2), bin(5 | 2))       # 7

bật bit 0: ans = 1 = 0b1
bật bit 3: ans = 9 = 0b1001
7 0b101 0b10 0b111


### 3.5. Đếm "có bao nhiêu phần tử có bit `b` bằng 1"

Với mỗi bit `b`, `sum((x >> b) & 1 for x in nums)` cho **số phần tử có bit `b` bật** (vì mỗi phần tử đóng góp `0` hoặc `1`). Đây chính là điều kiện của đề: bit `b` của K-or bật khi số đếm `>= k`.

In [6]:
nums = [7, 12, 9, 8, 9, 15]
for b in range(4):
    count = sum((x >> b) & 1 for x in nums)
    print(f"bit {b}: có {count} phần tử bật")

bit 0: có 4 phần tử bật
bit 1: có 2 phần tử bật
bit 2: có 3 phần tử bật
bit 3: có 5 phần tử bật


### 3.6. Giá trị tối đa: bao nhiêu bit cần duyệt?

`nums[i] < 2³¹` nên mọi số đều có tối đa **31 bit** (bit `0` đến bit `30`). Ta duyệt `range(31)`. (Python không giới hạn độ lớn số nguyên nên không tràn; dùng `31` đủ cho miền của đề.) Chú ý: `range(30)` chỉ tới bit `29`, **bỏ sót** bit `30`.

In [7]:
print(2**31 - 1, bin(2**31 - 1), (2**31 - 1).bit_length())   # 31 bit: tất cả 1
print(max(nums).bit_length(), "bit cần cho nums mẫu")
print((1 << 30), "= bit 30; range(30) không bao gồm nó:", 30 in range(30), "| range(31):", 30 in range(31))

2147483647 0b1111111111111111111111111111111 31
4 bit cần cho nums mẫu
1073741824 = bit 30; range(30) không bao gồm nó: False | range(31): True


### 3.7. K-or tổng quát hóa OR và AND

- `k = 1`: bit bật khi **ít nhất một** phần tử bật → đó là **OR** của mọi phần tử.
- `k = n` (`n = len(nums)`): bit bật khi **mọi** phần tử bật → đó là **AND** của mọi phần tử.

Với `functools.reduce`, ta gộp cả danh sách bằng một phép toán: `reduce(or_, nums)` hoặc `reduce(and_, nums)`.

In [8]:
print(reduce(or_, [10, 8, 5, 9, 11, 6, 8]))       # 15 (ví dụ 3)
print(reduce(and_, [2, 12, 1, 11, 4, 5]))         # 0  (ví dụ 2)
print(reduce(or_, [1, 2, 4]), reduce(and_, [7, 5, 6]))     # 7 4

15
0
7 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy viết các số ở dạng nhị phân và **đếm theo từng cột bit**. Với `nums = [7, 12, 9, 8, 9, 15]`, `k = 4`:

| số | bit 3 | bit 2 | bit 1 | bit 0 |
|---|---|---|---|---|
| 7 | 0 | 1 | 1 | 1 |
| 12 | 1 | 1 | 0 | 0 |
| 9 | 1 | 0 | 0 | 1 |
| 8 | 1 | 0 | 0 | 0 |
| 9 | 1 | 0 | 0 | 1 |
| 15 | 1 | 1 | 1 | 1 |
| **số bit 1 mỗi cột** | **5** | 3 | 2 | **4** |
| `>= k = 4`? | **có** | không | không | **có** |

Cột bit 3 và bit 0 đạt, nên K-or `= 1000₂ + 0001₂ = 1001₂ = 9`. Thuật toán: **với mỗi cột bit, đếm số phần tử có bit 1; nếu đếm `>= k` thì bật bit đó trong kết quả**.

## 5. Cách 1 — Brute Force / baseline: duyệt từng bit, đếm qua mọi phần tử

**Ý tưởng.** Với mỗi bit `b` từ `0` đến `30`: đếm `count = Σ (x >> b) & 1` qua mọi `x`; nếu `count >= k` thì `ans |= 1 << b`. Đây là mô phỏng trực tiếp định nghĩa của đề.

In [9]:
def find_k_or_bitwise(nums: List[int], k: int) -> int:
    ans = 0
    for b in range(31):                                  # bit 0 .. bit 30
        count = 0
        for x in nums:
            count += (x >> b) & 1                        # 1 nếu bit b của x bật
        if count >= k:                                   # ít nhất k phần tử bật
            ans |= 1 << b
    return ans


for nums, k, expected in examples:
    print(nums, k, "->", find_k_or_bitwise(nums, k), "(kỳ vọng", expected, ")")

[7, 12, 9, 8, 9, 15] 4 -> 9 (kỳ vọng 9 )
[2, 12, 1, 11, 4, 5] 6 -> 0 (kỳ vọng 0 )
[10, 8, 5, 9, 11, 6, 8] 1 -> 15 (kỳ vọng 15 )


In [10]:
# Từng bit của ví dụ 1
nums, k = [7, 12, 9, 8, 9, 15], 4
ans = 0
for b in range(5):
    count = sum((x >> b) & 1 for x in nums)
    ok = count >= k
    if ok:
        ans |= 1 << b
    print(f"bit {b}: count={count} {'>=' if ok else '<'} k={k} -> {'bật' if ok else 'tắt'} | ans={ans} ({bin(ans)})")

bit 0: count=4 >= k=4 -> bật | ans=1 (0b1)
bit 1: count=2 < k=4 -> tắt | ans=1 (0b1)
bit 2: count=3 < k=4 -> tắt | ans=1 (0b1)
bit 3: count=5 >= k=4 -> bật | ans=9 (0b1001)
bit 4: count=0 < k=4 -> tắt | ans=9 (0b1001)


**Độ phức tạp (Cách 1).** Gọi `n` là độ dài `nums`, `B = 31` là số bit. Hai vòng lồng nhau: **Time `O(B · n)`** (tức `O(n)` vì `B` là hằng số); chỉ vài biến: **Space `O(1)`**.

**Nút thắt.** Với mỗi `(bit, phần tử)` ta làm một phép dịch bit dù phần lớn bit là `0`. Ta có thể chỉ chạm tới các bit **đang bật** của từng số.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai hướng cải tiến:

1. **Chỉ duyệt các bit đang bật.** Với mỗi số `x`, thay vì xét cả 31 bit, ta lấy lần lượt **bit `1` thấp nhất** (`x & -x`), tăng bộ đếm của vị trí bit đó, rồi xóa bit ấy. Số bước bằng **số bit `1`** của `x`.
2. **Quan sát đặc biệt.** `k = 1` là **OR** và `k = n` là **AND** của mọi phần tử: có thể tính bằng `reduce` mà không cần đếm.

Cả hai đều cho cùng kết quả như định nghĩa; hướng 1 là cách đếm tổng quát gọn hơn, hướng 2 là lối tắt cho hai trường hợp biên.

## 7. Các cách tiếp theo

### Cách 2 — Đếm theo từng số: chỉ chạm các bit đang bật

**Giải thích.** Dùng mảng `cnt[0..30]`. Với mỗi `x`: lặp `while x`: `low = x & -x` (số chỉ còn bit `1` thấp nhất), `b = low.bit_length() - 1` (vị trí của nó), `cnt[b] += 1`, rồi `x ^= low` (xóa bit đó). Cuối cùng bật các bit `b` có `cnt[b] >= k`.

In [11]:
def find_k_or_lowbit(nums: List[int], k: int) -> int:
    cnt = [0] * 31                                  # cnt[b] = số phần tử có bit b bật
    for x in nums:
        while x:
            low = x & -x                            # chỉ giữ bit 1 thấp nhất của x
            cnt[low.bit_length() - 1] += 1          # vị trí bit = bit_length - 1
            x ^= low                                # xóa bit đó
    ans = 0
    for b in range(31):
        if cnt[b] >= k:
            ans |= 1 << b
    return ans


for nums, k, expected in examples:
    print(nums, k, "->", find_k_or_lowbit(nums, k), "(kỳ vọng", expected, ")")

[7, 12, 9, 8, 9, 15] 4 -> 9 (kỳ vọng 9 )
[2, 12, 1, 11, 4, 5] 6 -> 0 (kỳ vọng 0 )
[10, 8, 5, 9, 11, 6, 8] 1 -> 15 (kỳ vọng 15 )


In [12]:
# Quan sát cách tách bit của số 12 = 1100
x = 12
while x:
    low = x & -x
    print(f"x={bin(x):>6} low={bin(low):>6} vị trí bit={low.bit_length() - 1}")
    x ^= low

x=0b1100 low= 0b100 vị trí bit=2
x=0b1000 low=0b1000 vị trí bit=3


**Độ phức tạp Cách 2:** **Time `O(n · b̄)`** với `b̄` là số bit `1` trung bình mỗi số (tối đa 31) cộng `O(B)` bước cuối, **Space `O(B) = O(1)`**.

### Cách 3 — Chuỗi nhị phân đệm đủ 31 ký tự rồi `zip(*cột)`

**Giải thích.** Mỗi số đổi thành chuỗi nhị phân **đệm `0` đủ 31 ký tự** (`format(x, "031b")`); ký tự thứ `j` của chuỗi ứng với bit `30 - j`. `zip(*chuỗi)` **chuyển vị** thành các **cột** (mỗi cột là một bit qua mọi số); đếm `'1'` trong mỗi cột. Rất trực quan, giống bảng ở mục 4.

In [13]:
def find_k_or_columns(nums: List[int], k: int) -> int:
    rows = [format(x, "031b") for x in nums]        # mỗi số: chuỗi nhị phân đúng 31 ký tự
    ans = 0
    for j, column in enumerate(zip(*rows)):          # column = ký tự thứ j của mọi số
        if column.count("1") >= k:
            ans |= 1 << (30 - j)                     # ký tự j <-> bit 30 - j
    return ans


for nums, k, expected in examples:
    print(nums, k, "->", find_k_or_columns(nums, k), "(kỳ vọng", expected, ")")

[7, 12, 9, 8, 9, 15] 4 -> 9 (kỳ vọng 9 )
[2, 12, 1, 11, 4, 5] 6 -> 0 (kỳ vọng 0 )
[10, 8, 5, 9, 11, 6, 8] 1 -> 15 (kỳ vọng 15 )


**Độ phức tạp Cách 3:** **Time `O(B · n)`**, **Space `O(B · n)`** cho các chuỗi nhị phân.

### Cách 4 — Lối tắt cho `k = 1` (OR) và `k = n` (AND), còn lại đếm

**Giải thích.** Nếu `k == 1` trả `reduce(or_, nums)`; nếu `k == len(nums)` trả `reduce(and_, nums)`; trường hợp khác dùng cách đếm tổng quát. Đây là lối tắt hay để **kiểm chứng** kết quả (đáp án với `k = 1` phải đúng bằng OR, với `k = n` đúng bằng AND).

In [14]:
def find_k_or_shortcut(nums: List[int], k: int) -> int:
    if k == 1:
        return reduce(or_, nums)                     # ít nhất 1 phần tử bật = OR
    if k == len(nums):
        return reduce(and_, nums)                    # mọi phần tử bật = AND
    return find_k_or_bitwise(nums, k)                # tổng quát


for nums, k, expected in examples:
    print(nums, k, "->", find_k_or_shortcut(nums, k), "(kỳ vọng", expected, ")")

[7, 12, 9, 8, 9, 15] 4 -> 9 (kỳ vọng 9 )
[2, 12, 1, 11, 4, 5] 6 -> 0 (kỳ vọng 0 )
[10, 8, 5, 9, 11, 6, 8] 1 -> 15 (kỳ vọng 15 )


In [15]:
# Code nộp LeetCode cho Cách 1 (đơn giản, đủ nhanh, không cần lối tắt)
from typing import List

class Solution:
    def findKOr(self, nums: List[int], k: int) -> int:
        ans = 0
        for b in range(31):
            if sum((x >> b) & 1 for x in nums) >= k:
                ans |= 1 << b
        return ans

**Độ phức tạp Cách 4:** **Time `O(n)`** cho `k = 1` hoặc `k = n`; ngược lại `O(B · n)`; **Space `O(1)`**.

> Cách 1 (đếm theo từng bit) là cách nên nộp: ngắn, đúng với mọi `k`. Cách 2 đẹp về mặt kỹ thuật bit; Cách 3 trực quan; Cách 4 dùng để hiểu "K-or tổng quát OR/AND".

## 8. TRACE từng bước

Ta theo dõi (a) Cách 1 với `nums = [7, 12, 9, 8, 9, 15]`, `k = 4`; (b) Cách 2 tách bit từng số.

In [16]:
def trace_bitwise(nums, k, bits=5):
    ans = 0
    print("nums =", nums, "| k =", k)
    for b in range(bits):
        column = [(x >> b) & 1 for x in nums]
        count = sum(column)
        ok = count >= k
        if ok:
            ans |= 1 << b
        print(f"bit {b}: cột={column} count={count} -> {'>= k: bật' if ok else '< k: tắt'} | ans={ans:>3} {bin(ans)}")
    print("=> K-or =", ans)
    return ans


assert trace_bitwise([7, 12, 9, 8, 9, 15], 4) == 9

nums = [7, 12, 9, 8, 9, 15] | k = 4
bit 0: cột=[1, 0, 1, 0, 1, 1] count=4 -> >= k: bật | ans=  1 0b1
bit 1: cột=[1, 0, 0, 0, 0, 1] count=2 -> < k: tắt | ans=  1 0b1
bit 2: cột=[1, 1, 0, 0, 0, 1] count=3 -> < k: tắt | ans=  1 0b1
bit 3: cột=[0, 1, 1, 1, 1, 1] count=5 -> >= k: bật | ans=  9 0b1001
bit 4: cột=[0, 0, 0, 0, 0, 0] count=0 -> < k: tắt | ans=  9 0b1001
=> K-or = 9


In [17]:
def trace_lowbit(nums, k, bits=5):
    cnt = [0] * bits
    for x in nums:
        original, parts = x, []
        while x:
            low = x & -x
            b = low.bit_length() - 1
            cnt[b] += 1
            parts.append(b)
            x ^= low
        print(f"số {original:>2} = {bin(original):>7}: các bit bật {parts} -> cnt={cnt}")
    ans = sum(1 << b for b in range(bits) if cnt[b] >= k)
    print(f"cnt = {cnt}, k = {k} -> K-or = {ans}")
    return ans


assert trace_lowbit([7, 12, 9, 8, 9, 15], 4) == 9

số  7 =   0b111: các bit bật [0, 1, 2] -> cnt=[1, 1, 1, 0, 0]
số 12 =  0b1100: các bit bật [2, 3] -> cnt=[1, 1, 2, 1, 0]
số  9 =  0b1001: các bit bật [0, 3] -> cnt=[2, 1, 2, 2, 0]
số  8 =  0b1000: các bit bật [3] -> cnt=[2, 1, 2, 3, 0]
số  9 =  0b1001: các bit bật [0, 3] -> cnt=[3, 1, 2, 4, 0]
số 15 =  0b1111: các bit bật [0, 1, 2, 3] -> cnt=[4, 2, 3, 5, 0]
cnt = [4, 2, 3, 5, 0], k = 4 -> K-or = 9


Bảng tóm tắt (Cách 1, `k = 4`):

| bit `b` | cột bit (7,12,9,8,9,15) | `count` | `>= 4`? | `ans` |
|---|---|---|---|---|
| 0 | 1,0,1,0,1,1 | 4 | có | 1 |
| 1 | 1,0,0,0,0,1 | 2 | không | 1 |
| 2 | 1,1,0,0,0,1 | 3 | không | 1 |
| 3 | 0,1,1,1,1,1 | 5 | có | 9 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 1 (đếm từng bit). Đúng class `Solution`, đúng method `findKOr(self, nums: List[int], k: int) -> int`.

In [18]:
from typing import List

class Solution:
    def findKOr(self, nums: List[int], k: int) -> int:
        ans = 0
        for b in range(31):                              # nums[i] < 2^31 nên có tối đa bit 0..30
            count = sum((x >> b) & 1 for x in nums)      # số phần tử có bit b bật
            if count >= k:                               # ít nhất k phần tử
                ans |= 1 << b                            # bật bit b trong kết quả
        return ans


sol = Solution()
for nums, k, expected in examples:
    print(nums, k, "->", sol.findKOr(nums, k))

[7, 12, 9, 8, 9, 15] 4 -> 9
[2, 12, 1, 11, 4, 5] 6 -> 0
[10, 8, 5, 9, 11, 6, 8] 1 -> 15


## 10. Vì sao thuật toán đúng

- **Đúng định nghĩa:** bit `b` của K-or bật khi và chỉ khi số phần tử có bit `b` bật là `>= k`. Thuật toán tính đúng số phần tử đó (`count`) và bật bit `b` đúng khi `count >= k`.
- **Các bit độc lập:** K-or được xác định **từng bit một**, không bit nào phụ thuộc bit khác, nên duyệt từng bit rồi ghép bằng `|` cho đúng.
- **Đủ bit:** vì `nums[i] < 2³¹`, mọi bit từ `31` trở lên của mọi `x` đều `0`, nên `count = 0 < k` (do `k >= 1`) và các bit đó không bật. Duyệt `0..30` là đủ.
- **Trường hợp đặc biệt khớp:** `k = 1` cho `count >= 1` ⇔ có ít nhất một phần tử bật ⇔ OR; `k = n` cho `count >= n` ⇔ mọi phần tử bật ⇔ AND.
- **Tính chất kiểm được:** K-or tăng không nghiêm ngặt khi `k` giảm: `kor(k + 1) ⊆ kor(k)` (là tập con theo bit), tức `kor(k + 1) & kor(k) == kor(k + 1)`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài `nums`, `B = 31` là số bit.

- **Cách 1:** Time `O(B · n)`, tức `O(n)` với `B` hằng số; Space `O(1)`.
- **Cách 2:** Time `O(Σ popcount(x))` cộng `O(B)`, Space `O(B)`.
- **Cách 3:** Time `O(B · n)`, Space `O(B · n)` (chuỗi nhị phân).
- **Cách 4:** Time `O(n)` ở hai trường hợp biên, ngược lại `O(B · n)`.
- Mọi cách phải đọc mọi phần tử nên không thể tốt hơn `O(n)`. Với `n <= 50` và `B = 31` chỉ khoảng 1 550 phép.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` rồi kiểm tra bằng `assert`.

In [19]:
sol = Solution()
impls = {
    "bitwise": find_k_or_bitwise,
    "lowbit": find_k_or_lowbit,
    "columns": find_k_or_columns,
    "shortcut": find_k_or_shortcut,
    "submit": sol.findKOr,
}

for nums, k, expected in examples:
    for key, fn in impls.items():
        assert fn(list(nums), k) == expected, (key, nums, k, fn(list(nums), k), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [20]:
def check_all(nums, k, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums), k)
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([0], 1, 0, "một phần tử bằng 0")
check_all([5], 1, 5, "một phần tử: K-or là chính nó")
check_all([0, 0, 0], 1, 0, "toàn 0")
check_all([1, 2, 4], 1, 7, "k = 1 là OR: 1 | 2 | 4")
check_all([1, 2, 4], 3, 0, "k = n là AND: 1 & 2 & 4 = 0")
check_all([7, 7, 7], 3, 7, "k = n, mọi số giống nhau")
check_all([3, 3, 5], 2, 3 | 1, "k = 2: bit 0 có 3 phần tử, bit 1 có 2, bit 2 có 1")
check_all([2**31 - 1, 2**31 - 1], 2, 2**31 - 1, "số lớn nhất của đề: 31 bit bật")
check_all([2**30, 2**30 + 1, 0], 2, 2**30, "bit 30 (dễ bị bỏ sót nếu chỉ duyệt 30 bit)")

PASS một phần tử bằng 0 -> 0
PASS một phần tử: K-or là chính nó -> 5
PASS toàn 0 -> 0
PASS k = 1 là OR: 1 | 2 | 4 -> 7
PASS k = n là AND: 1 & 2 & 4 = 0 -> 0
PASS k = n, mọi số giống nhau -> 7
PASS k = 2: bit 0 có 3 phần tử, bit 1 có 2, bit 2 có 1 -> 3
PASS số lớn nhất của đề: 31 bit bật -> 2147483647
PASS bit 30 (dễ bị bỏ sót nếu chỉ duyệt 30 bit) -> 1073741824


### 12.2. Tính chất và phản ví dụ

In [21]:
# Tính chất 1: k = 1 <-> OR, k = n <-> AND
random.seed(11)
for _ in range(200):
    n = random.randint(1, 50)
    nums = [random.randrange(2**31) for _ in range(n)]
    assert sol.findKOr(nums, 1) == reduce(or_, nums)
    assert sol.findKOr(nums, n) == reduce(and_, nums)
print("PASS k = 1 cho OR; k = n cho AND")

# Tính chất 2: kor(k + 1) là tập con (theo bit) của kor(k)
for _ in range(200):
    n = random.randint(2, 30)
    nums = [random.randrange(2**31) for _ in range(n)]
    k = random.randint(1, n - 1)
    a, b = sol.findKOr(nums, k), sol.findKOr(nums, k + 1)
    assert b & a == b
print("PASS K-or giảm dần theo k (bao hàm theo bit)")

# Tính chất 3: mọi bit của kết quả có mặt ở ít nhất k phần tử; mọi bit tắt có mặt ở < k phần tử
nums = [random.randrange(2**31) for _ in range(40)]
k = 7
res = sol.findKOr(nums, k)
for b in range(31):
    c = sum((x >> b) & 1 for x in nums)
    assert ((res >> b) & 1 == 1) == (c >= k)
print("PASS bit bật <=> có ít nhất k phần tử bật")

# Đầu vào không bị sửa
nums = [7, 12, 9, 8, 9, 15]
copy = list(nums)
for key, fn in impls.items():
    fn(nums, 4)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: chỉ duyệt 30 bit (bỏ sót bit 30)
def sai_30_bit(nums, k):
    ans = 0
    for b in range(30):
        if sum((x >> b) & 1 for x in nums) >= k:
            ans |= 1 << b
    return ans

assert sai_30_bit([2**30, 2**30], 2) == 0 and sol.findKOr([2**30, 2**30], 2) == 2**30
print("PASS phản ví dụ: range(30) bỏ sót bit 30")

# Phản ví dụ 2: dùng > k thay vì >= k
def sai_lon_hon(nums, k):
    ans = 0
    for b in range(31):
        if sum((x >> b) & 1 for x in nums) > k:
            ans |= 1 << b
    return ans

assert sai_lon_hon([7, 12, 9, 8, 9, 15], 4) != 9
print("PASS phản ví dụ: > k cho", sai_lon_hon([7, 12, 9, 8, 9, 15], 4), "thay vì 9")

PASS k = 1 cho OR; k = n cho AND
PASS K-or giảm dần theo k (bao hàm theo bit)
PASS bit bật <=> có ít nhất k phần tử bật
PASS đầu vào không bị sửa
PASS phản ví dụ: range(30) bỏ sót bit 30
PASS phản ví dụ: > k cho 8 thay vì 9


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [22]:
random.seed(2917)
for trial in range(2000):
    n = random.randint(1, 50)
    big = random.choice([2, 16, 1024, 2**31])
    nums = [random.randrange(big) for _ in range(n)]
    k = random.randint(1, n)
    ref = find_k_or_bitwise(list(nums), k)
    for key, fn in impls.items():
        assert fn(list(nums), k) == ref, (key, nums, k, fn(list(nums), k), ref)
print("PASS 2000 bộ ngẫu nhiên - mọi cách khớp nhau")

PASS 2000 bộ ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ duyệt 30 bit (bỏ sót bit 30)

In [23]:
def sai_30(nums, k):
    ans = 0
    for b in range(30):                               # thiếu bit 30
        if sum((x >> b) & 1 for x in nums) >= k:
            ans |= 1 << b
    return ans


nums = [2**30 + 5, 2**30 + 1]
print("sai  :", sai_30(nums, 2), "| đúng:", Solution().findKOr(nums, 2))

sai  : 1 | đúng: 1073741825


**Vì sao sai:** `nums[i] < 2³¹` nên bit lớn nhất có thể bật là bit `30` (giá trị `2³⁰`). `range(30)` chỉ tới bit `29`, nên bit `30` không bao giờ được xét. Hãy dùng `range(31)`.

### Lỗi 2: Dùng `> k` thay vì `>= k`

In [24]:
def sai_lon_hon(nums, k):
    ans = 0
    for b in range(31):
        if sum((x >> b) & 1 for x in nums) > k:
            ans |= 1 << b
    return ans


print("sai  :", sai_lon_hon([7, 12, 9, 8, 9, 15], 4), "| đúng:", Solution().findKOr([7, 12, 9, 8, 9, 15], 4))
print("sai  :", sai_lon_hon([7, 7, 7], 3), "(k = n: không bao giờ đủ) | đúng:", Solution().findKOr([7, 7, 7], 3))

sai  : 8 | đúng: 9
sai  : 0 (k = n: không bao giờ đủ) | đúng: 7


**Vì sao sai:** đề nói **ít nhất `k`** nên điều kiện là `count >= k`. Với `k = n` thì `count > k` **không bao giờ** đúng (vì `count <= n`), nên mọi bit đều tắt dù cả `n` phần tử cùng bật bit đó.

### Lỗi 3: Cộng `x & (1 << b)` vào bộ đếm

In [25]:
def sai_cong_mat_na(nums, k):
    ans = 0
    for b in range(31):
        count = 0
        for x in nums:
            count += x & (1 << b)                      # cộng 2^b, không phải 1
        if count >= k:
            ans |= 1 << b
    return ans


print("sai  :", sai_cong_mat_na([7, 12, 9, 8, 9, 15], 4), "| đúng:", Solution().findKOr([7, 12, 9, 8, 9, 15], 4))

sai  : 15 | đúng: 9


**Vì sao sai:** `x & (1 << b)` là `0` hoặc **`2ᵇ`**, không phải `0` hoặc `1`. Cộng `2ᵇ` làm bộ đếm phình lên. Phải dịch về bit thấp nhất `(x >> b) & 1` hoặc chuyển thành bool: `1 if x & (1 << b) else 0`.

### Lỗi 4: Cộng `b` thay vì `1 << b` vào kết quả

In [26]:
def sai_cong_chi_so(nums, k):
    ans = 0
    for b in range(31):
        if sum((x >> b) & 1 for x in nums) >= k:
            ans += b                                   # cộng chỉ số bit, không phải giá trị bit
    return ans


print("sai  :", sai_cong_chi_so([7, 12, 9, 8, 9, 15], 4), "| đúng:", Solution().findKOr([7, 12, 9, 8, 9, 15], 4))

sai  : 3 | đúng: 9


**Vì sao sai:** bit `b` có **giá trị `2ᵇ`**, không phải `b`. Phải dùng `ans |= 1 << b` (hoặc `ans += 1 << b`). Ở ví dụ: bit `0` và `3` bật cho `1 + 8 = 9`, còn cộng chỉ số cho `0 + 3 = 3`.

### Lỗi 5: Chuỗi nhị phân không đệm cùng độ dài khi dùng `zip`

In [27]:
def sai_khong_dem(nums, k):
    rows = [bin(x)[2:] for x in nums]                  # độ dài khác nhau!
    ans = 0
    for j, column in enumerate(zip(*rows)):            # zip dừng ở chuỗi ngắn nhất, và căn LỆCH bên trái
        if column.count("1") >= k:
            ans |= 1 << (len(rows[0]) - 1 - j)
    return ans


print("sai  :", sai_khong_dem([7, 12, 9, 8, 9, 15], 4), "| đúng:", Solution().findKOr([7, 12, 9, 8, 9, 15], 4))

sai  : 4 | đúng: 9


**Vì sao sai:** chuỗi `bin(x)[2:]` có độ dài **khác nhau** (`7 -> '111'`, `12 -> '1100'`), nên các ký tự cùng vị trí **không cùng bit**; `zip` còn cắt theo chuỗi ngắn nhất. Phải **đệm `0` bên trái** cho cùng độ dài: `format(x, "031b")` hoặc `bin(x)[2:].zfill(31)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "bit thứ `i` bằng 1 khi ... ít nhất `k` phần tử" → **đếm theo từng cột bit**.
- Công thức nhớ: `for b in range(31): if sum((x >> b) & 1 for x in nums) >= k: ans |= 1 << b`.
- Nhận ra **OR** (`k = 1`) và **AND** (`k = n`) là hai trường hợp biên để tự kiểm tra.
- Số trong đề `< 2³¹` → duyệt `31` bit (`0..30`); với `< 2³²` thì `32` bit.
- Khi cần duyệt các bit đang bật: `low = x & -x`, `x ^= low`, vị trí `= low.bit_length() - 1`.
- Điều kiện "ít nhất" → `>=`; "nhiều hơn" → `>`; đọc kỹ từng chữ.

## 15. Biến thể của bài

1. **Bit thứ `i` bật khi có đúng `k` phần tử bật**: thay `>= k` bằng `== k`.
2. **Bit bật khi số phần tử bật là chẵn / lẻ** (phép XOR nhiều số: parity).
3. **Bit bật khi quá nửa** số phần tử bật (majority bit): `count > n // 2`.
4. **K-and**: bit bật khi có ít nhất `k` phần tử **tắt**... (đối xứng, đếm bit `0`).
5. Tìm **giá trị `k` lớn nhất** để K-or khác `0`.
6. Dãy các K-or với mọi `k = 1..n` (mỗi `k` một kết quả), tính cùng lúc nhờ đếm một lần.

Ta thử biến thể 3 và 6:

In [28]:
def majority_bits(nums):
    n = len(nums)
    ans = 0
    for b in range(31):
        if sum((x >> b) & 1 for x in nums) > n // 2:
            ans |= 1 << b
    return ans


def all_k_ors(nums):
    """Đếm một lần rồi trả K-or cho mọi k = 1..n."""
    cnt = [sum((x >> b) & 1 for x in nums) for b in range(31)]
    return [sum(1 << b for b in range(31) if cnt[b] >= k) for k in range(1, len(nums) + 1)]


print("3. bit đa số của [7, 12, 9, 8, 9, 15]:", majority_bits([7, 12, 9, 8, 9, 15]))
res = all_k_ors([7, 12, 9, 8, 9, 15])
print("6. K-or cho k = 1..6:", res)
assert res[3] == 9 and all(res[k - 1] == Solution().findKOr([7, 12, 9, 8, 9, 15], k) for k in range(1, 7))

3. bit đa số của [7, 12, 9, 8, 9, 15]: 9
6. K-or cho k = 1..6: [15, 15, 13, 9, 8, 0]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm từng bit qua mọi phần tử | `sum((x >> b) & 1)` rồi `ans \|= 1 << b` | `O(B·n)` | `O(1)` | **cách nộp bài**, đơn giản và đúng với mọi `k` |
| 2. `low = x & -x` chỉ chạm bit bật | bộ đếm `cnt[b]` | `O(Σ popcount)` | `O(B)` | muốn học kỹ thuật bit, số có ít bit 1 |
| 3. Chuỗi nhị phân `zip(*cột)` | nhìn theo cột như bảng | `O(B·n)` | `O(B·n)` | trực quan, giảng dạy |
| 4. Lối tắt OR/AND + tổng quát | `reduce(or_)`, `reduce(and_)` | `O(n)` (hai biên) | `O(1)` | kiểm chứng; hiểu K-or tổng quát hóa OR/AND |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(13 >> 2) & 1`, `1 << 5`, `6 & (1 << 2)`, `5 | 2`, `Solution().findKOr([1, 2, 4], 1)`.

**Bài 2 (tính tay).** Với `nums = [3, 5, 6, 7]` và `k = 2`: viết nhị phân 3 bit của từng số, đếm số bit `1` mỗi cột, rồi tính K-or. Làm lại với `k = 3`.

**Bài 3 (điền TODO).** Hoàn thành `find_k_or_todo(nums, k)` theo Cách 1.

In [29]:
def find_k_or_todo(nums, k):
    ans = 0
    for b in range(31):
        # TODO 1: count = số phần tử x có bit b bật, dùng (x >> b) & 1
        # TODO 2: nếu count >= k thì bật bit b trong ans: ans |= 1 << b
        pass
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `find_k_or_lowbit2(nums, k)` bằng kỹ thuật `low = x & -x` (Cách 2) mà không nhìn code mẫu. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `find_exact_k_or(nums, k)`: bit bật khi có **đúng `k`** phần tử bật. (b) Viết `largest_k_nonzero(nums)` trả về **`k` lớn nhất** mà K-or khác `0` (trả `0` nếu với mọi `k` K-or đều bằng `0`, ví dụ mọi phần tử bằng `0`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `13 = 1101` nên bit 2 là `1`; `32`; `6 = 110` có bit 2 nên `6 & 4 = 4`; `7`; `7`.
- **Mức 2 (vừa):** Bài 2: `3 = 011`, `5 = 101`, `6 = 110`, `7 = 111`: cột bit 2 có 3 số bật (`5, 6, 7`), cột bit 1 có 3 (`3, 6, 7`), cột bit 0 có 3 (`3, 5, 7`). Với `k = 2` mọi cột `>= 2`: K-or `= 7`; với `k = 3` mọi cột `>= 3`: K-or `= 7`. (Thử `k = 4` thì mọi cột `< 4`: `0`.) Bài 3: hai TODO đã ghi trong comment. Bài 4: xem Cách 2.
- **Mức 3 (sâu):** Bài 5(a): đổi `>=` thành `==`. Bài 5(b): tính `cnt[b]` một lần; `k` lớn nhất mà tồn tại bit có `cnt[b] >= k` chính là `max(cnt)`; nếu `max(cnt) == 0` trả `0`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert (13 >> 2) & 1 == 1 and (1 << 5) == 32
assert 6 & (1 << 2) == 4 and (5 | 2) == 7
assert Solution().findKOr([1, 2, 4], 1) == 7

# Bài 2
assert Solution().findKOr([3, 5, 6, 7], 2) == 7
assert Solution().findKOr([3, 5, 6, 7], 3) == 7
assert Solution().findKOr([3, 5, 6, 7], 4) == 3 & 5 & 6 & 7 == 0

# Bài 3
def find_k_or_todo_ans(nums, k):
    ans = 0
    for b in range(31):
        count = sum((x >> b) & 1 for x in nums)
        if count >= k:
            ans |= 1 << b
    return ans

for nums, k, expected in examples:
    assert find_k_or_todo_ans(nums, k) == expected

# Bài 4
def find_k_or_lowbit2(nums, k):
    cnt = [0] * 31
    for x in nums:
        while x:
            low = x & -x
            cnt[low.bit_length() - 1] += 1
            x ^= low
    return sum(1 << b for b in range(31) if cnt[b] >= k)

for nums, k, expected in examples:
    assert find_k_or_lowbit2(nums, k) == expected

# Bài 5(a)
def find_exact_k_or(nums, k):
    ans = 0
    for b in range(31):
        if sum((x >> b) & 1 for x in nums) == k:
            ans |= 1 << b
    return ans

assert find_exact_k_or([7, 12, 9, 8, 9, 15], 2) == 2          # chỉ bit 1 có đúng 2 phần tử
assert find_exact_k_or([1, 1, 1], 3) == 1

# Bài 5(b)
def largest_k_nonzero(nums):
    cnt = [sum((x >> b) & 1 for x in nums) for b in range(31)]
    return max(cnt)

assert largest_k_nonzero([7, 12, 9, 8, 9, 15]) == 5           # bit 3 có 5 phần tử
assert largest_k_nonzero([0, 0]) == 0
for k in range(1, 7):
    assert (Solution().findKOr([7, 12, 9, 8, 9, 15], k) != 0) == (k <= largest_k_nonzero([7, 12, 9, 8, 9, 15]))
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm theo từng bit, bật bit khi đủ `k` phần tử)

```python
from typing import List

class Solution:
    def findKOr(self, nums: List[int], k: int) -> int:
        ans = 0
        for b in range(31):
            if sum((x >> b) & 1 for x in nums) >= k:
                ans |= 1 << b
        return ans
```

**Checklist**

- [ ] Duyệt **31 bit** (`range(31)`), không bỏ sót bit 30?
- [ ] Điều kiện `count >= k` (ít nhất `k`), không phải `> k`?
- [ ] Đếm bằng `(x >> b) & 1` (0/1), không cộng `x & (1 << b)`?
- [ ] Kết quả dựng bằng `ans |= 1 << b` (giá trị bit), không `ans += b`?
- [ ] Nếu dùng chuỗi nhị phân: đã **đệm đủ độ dài** (`format(x, "031b")`)?
- [ ] Đã thử `k = 1` (OR) và `k = n` (AND), số `0`, số lớn nhất `2^31 - 1`?
- [ ] Skeleton đúng `class Solution` / `def findKOr(self, nums: List[int], k: int) -> int`?